# 05_listening_patterns

DML: gold_listening_patterns — Listening patterns by hour and day of week.

In [ ]:
%run ../../tools/config/settings
%run ../../tools/delta/upsert

In [ ]:
dbutils.widgets.text("snapshot_date", "")
snapshot_date = dbutils.widgets.get("snapshot_date")

In [ ]:
from pyspark.sql import functions as F

fct   = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.fct_plays")
dim_t = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.dim_time").select("played_at", "play_hour", "day_of_week", "day_name")

result = (
    fct
    .join(dim_t, "played_at", "left")
    .groupBy("play_hour", "day_of_week", "day_name")
    .agg(F.count("play_id").alias("play_count"))
    .withColumnRenamed("play_hour", "hour_of_day")
    .withColumn("snapshot_date", F.to_date(F.lit(snapshot_date)))
    .select("snapshot_date", "hour_of_day", "day_of_week", "day_name", "play_count")
)

upsert_delta(result, f"{CATALOG}.{GOLD_SCHEMA}.gold_listening_patterns", ["snapshot_date", "hour_of_day", "day_of_week"])
display(result.orderBy("day_of_week", "hour_of_day"))